# 01 — Municipal election results 2011–2021

**Owner:** A  
**Goal:** one row per municipality per election with registered voters, votes cast, spoilt votes

**Reads:** `data/raw/iec_lge/<year>/`  
**Writes:** `INTERIM / "results_lge_2011_2021.csv"`

Run the two setup cells below first, every time.

## Setup

In [ ]:
import sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Thato20-M/democratic-funnel.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/democratic-funnel")
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO_DIR)], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "src" / "config.py").exists())

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("Repo:", REPO_DIR)

In [ ]:
from src import config
from src.config import RAW, INTERIM, PROCESSED, MODELS, FIGURES, SEED
from src.io_utils import read_any, clean_columns, inventory
from src.privacy import drop_personal

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import warnings; warnings.filterwarnings("ignore", category=FutureWarning)

config.make_dirs(); config.set_seed()
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid"); plt.rcParams["figure.dpi"] = 110
print(config.describe())

## Inspect what we downloaded

List the files and see the detected header row and columns for each sheet.

In [ ]:
inv = inventory(RAW / "iec_lge")
inv

## Read one year

Start with 2021. `read_any` skips the IEC title rows automatically.

In [ ]:
df21 = clean_columns(read_any(RAW / "iec_lge" / "2021" / "<file>.xlsx"))
print(df21.shape)
df21.head()

## Keep the PR ballot and the columns we need

Rename to our standard names: muni_code, muni_name, province, registered, votes_cast, spoilt.

In [ ]:
# df21 = df21[df21["ballot_type"].str.contains("PR", case=False, na=False)]
# df21 = df21.rename(columns={...})[["muni_code","muni_name","province",
#                                    "registered","votes_cast","spoilt"]]
# df21["election"] = 2021

## Repeat for 2016 and 2011, then stack

Column names differ between years — check each one before concatenating.

In [ ]:
# results = pd.concat([df11, df16, df21], ignore_index=True)

## Checks before saving

Municipality count per year, no missing codes, turnout between 0.2 and 0.9.

In [ ]:
# print(results.groupby("election")["muni_code"].nunique())
# print(results["muni_code"].isna().sum())
# results["turnout"] = results["votes_cast"] / results["registered"]
# print(results["turnout"].describe())

## Save output

In [ ]:
# out.to_csv(INTERIM / "results_lge_2011_2021.csv", index=False)
# print(out.shape); out.head()